<a href="https://colab.research.google.com/github/Bigoal/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigoal/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Two signal checks, then my rule

**Signal A — staleness (behind the refresh flags).** FlyRank's refresh-flag logic
(`scripts/02_baseline_score.py`, `stale_visible_page`) treats a page as "stale" once
`days_since_last_update >= 180`. Claim: stale pages decline more than fresh ones.
Test below: decline rate by `freshness_tier`, and by the flag's own `>=180` cut, each with n.

**Signal B — CTR vs. position (behind the CTR-fix logic).** The same script's
`low_ctr_visible_page` reason assumes a CTR that's too low *for the position* is a fixable
opportunity. Claim: CTR falls as position gets worse (the "CTR cliff"). Test below:
click-weighted CTR by `position_tier`, on pages with real visibility, with n.

**My rule, in plain words.** A page is worth a CTR review if searchers are actually seeing
it — real volume (`impressions_90d >= 500`) at a position where results get seen
(`avg_position` between 1 and 20) — but they aren't clicking (`ctr < 0.5`, i.e. under 0.5%).
That's a visible page with a click problem, not a ranking problem.

I dropped staleness from the rule: the check below comes back **OPPOSITE**, so wiring
`days_since_last_update` into the score would have coded a false belief straight into the
baseline. The CTR/position check comes back **CONFIRMED**, so that's what the rule leans on.

- **Reason code:** `low_ctr_visible_page` when the rule fires, `not_flagged` otherwise.
- **Action label:** `review_ctr` when it fires, `monitor` otherwise.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Bigoal/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # work/notebooks/ -> repo root

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Working dir:", os.getcwd())

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print(f"{len(df):,} pages | overall decline rate: {df['is_declining_label'].mean():.3f}\n")

# ---------------------------------------------------------------------------
# Signal A — staleness, behind the refresh flags
# ---------------------------------------------------------------------------
print("--- Signal A: decline rate by freshness_tier ---")
tier_order = ["0-30", "31-90", "91-180", "181+"]
tableA = (
    df.groupby("freshness_tier")["is_declining_label"]
    .agg(decline_rate="mean", n="count")
    .reindex(tier_order)
)
print(tableA.round(3))

print("\n--- Signal A, the flag's own cut: days_since_last_update >= 180 ---")
stale_flag = df["days_since_last_update"] >= 180
tableA2 = df.groupby(stale_flag)["is_declining_label"].agg(decline_rate="mean", n="count")
tableA2.index = ["not_stale (<180d)", "stale (>=180d)"]
print(tableA2.round(3))

verdict_a = "OPPOSITE"
print(
    f"\nVerdict A: {verdict_a} — pages past the flag's own 180-day cutoff decline LESS often "
    f"({tableA2.loc['stale (>=180d)','decline_rate']:.3f}, n={tableA2.loc['stale (>=180d)','n']:.0f}) "
    f"than pages under it ({tableA2.loc['not_stale (<180d)','decline_rate']:.3f}, "
    f"n={tableA2.loc['not_stale (<180d)','n']:.0f}). Both buckets clear the ~50-row floor, so this "
    "isn't noise from a tiny cell — staleness alone does not predict decline in this slice. "
    "It stays OUT of my rule."
)

# ---------------------------------------------------------------------------
# Signal B — CTR vs. position, behind the CTR-fix logic
# ---------------------------------------------------------------------------
print("\n--- Signal B: click-weighted CTR by position_tier (impressions_90d >= 100) ---")
pos_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
visible = df[df["impressions_90d"] >= 100]
tableB = (
    visible.groupby("position_tier")
    .apply(lambda d: pd.Series({
        "weighted_ctr_pct": d["clicks_90d"].sum() / d["impressions_90d"].sum() * 100,
        "n": len(d),
    }), include_groups=False)
    .reindex(pos_order)
)
print(tableB.round(3))

verdict_b = "CONFIRMED"
print(
    f"\nVerdict B: {verdict_b} — weighted CTR falls in step with position, from "
    f"{tableB.loc['top_3','weighted_ctr_pct']:.2f}% at top_3 (n={tableB.loc['top_3','n']:.0f}) "
    f"down to {tableB.loc['deep','weighted_ctr_pct']:.2f}% at deep (n={tableB.loc['deep','n']:.0f}). "
    "Every bucket clears the sample floor. CTR really is position-relative, so a CTR that's low "
    "FOR ITS POSITION is a real signal, not noise. It goes IN to my rule."
)

Working dir: /content/flyrank-ml-internship
30,000 pages | overall decline rate: 0.542

--- Signal A: decline rate by freshness_tier ---
                decline_rate      n
freshness_tier                     
0-30                   0.511  20480
31-90                  0.589    175
91-180                 0.611   9171
181+                   0.471    174

--- Signal A, the flag's own cut: days_since_last_update >= 180 ---
                   decline_rate      n
not_stale (<180d)         0.542  29826
stale (>=180d)            0.471    174

Verdict A: OPPOSITE — pages past the flag's own 180-day cutoff decline LESS often (0.471, n=174) than pages under it (0.542, n=29826). Both buckets clear the ~50-row floor, so this isn't noise from a tiny cell — staleness alone does not predict decline in this slice. It stays OUT of my rule.

--- Signal B: click-weighted CTR by position_tier (impressions_90d >= 100) ---
               weighted_ctr_pct       n
position_tier                          
top_3  

## 2. Build the ranked queue (writes the CSV)

Score = `impressions_90d` when a page is visible (`impressions_90d >= 500`), rankable
(`0 < avg_position <= 20`), and getting clicked less than expected for that (`ctr < 0.5`);
zero otherwise. No fitted weights, no partial credit — among flagged pages, the ones with the
biggest audience rank first, which is exactly what "review this one before that one" needs.

Inputs used: `impressions_90d`, `avg_position`, `ctr` — all trailing-90-day observed metrics.
Nothing from `trend_direction` / `trend_pct` (the label's own source) or any comparison-window
column (`*_last_30d`, `*_prev_30d`) goes into the score.

In [ ]:
visible = df["impressions_90d"] >= 500
good_position = (df["avg_position"] > 0) & (df["avg_position"] <= 20)
low_ctr = df["ctr"] < 0.5
flagged = visible & good_position & low_ctr

df["score"] = np.where(flagged, df["impressions_90d"], 0)
df["reason_code"] = np.where(flagged, "low_ctr_visible_page", "not_flagged")
df["action"] = np.where(flagged, "review_ctr", "monitor")
df["rank"] = df["score"].rank(method="first", ascending=False).astype(int)

queue_cols = [
    "rank", "content_id", "client_id", "score", "reason_code", "action",
    "impressions_90d", "avg_position", "position_tier", "ctr",
    "days_since_last_update", "content_type", "trend_direction", "is_declining_label",
]
queue = df.sort_values("rank")[queue_cols].reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
out_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(out_path, index=False)

print(f"Flagged {flagged.sum():,} of {len(df):,} pages ({flagged.mean():.1%}).")
print(f"Wrote {len(queue):,} ranked rows to {out_path}\n")

base_rate = df["is_declining_label"].mean()
flagged_rate = df.loc[flagged, "is_declining_label"].mean()
print(
    f"Decline rate check (context only, not a rule input): base = {base_rate:.3f} "
    f"(n={len(df):,}) vs flagged = {flagged_rate:.3f} (n={int(flagged.sum()):,})."
)

queue.head(10)

Flagged 9,759 of 30,000 pages (32.5%).
Wrote 30,000 ranked rows to work/outputs/baseline_action_score.csv

Decline rate check (context only, not a rule input): base = 0.542 (n=30,000) vs flagged = 0.627 (n=9,759).


,rank,content_id,client_id,score,reason_code,action,impressions_90d,avg_position,position_tier,ctr,days_since_last_update,content_type,trend_direction,is_declining_label
0,1,content_5fe46e04994d,client_4e07408562,517715,low_ctr_visible_page,review_ctr,517715,4.2,page_1,0.14,104,keyword article,down,1
1,2,content_aaef01a50def,client_19581e27de,517109,low_ctr_visible_page,review_ctr,517109,5.4,page_1,0.25,22,keyword article,stable,0
2,3,content_8c19996aa890,client_4e07408562,509252,low_ctr_visible_page,review_ctr,509252,2.5,top_3,0.15,20,keyword article,down,1
3,4,content_4c36c775b818,client_4e07408562,463103,low_ctr_visible_page,review_ctr,463103,2.3,top_3,0.41,20,keyword article,down,1
4,5,content_1a9e894be2e2,client_19581e27de,416180,low_ctr_visible_page,review_ctr,416180,4.0,page_1,0.23,22,keyword article,down,1
5,6,content_db5989a78dd3,client_4e07408562,345111,low_ctr_visible_page,review_ctr,345111,5.4,page_1,0.21,20,keyword article,up,0
6,7,content_cb112fce36be,client_19581e27de,309910,low_ctr_visible_page,review_ctr,309910,5.6,page_1,0.16,104,keyword article,down,1
7,8,content_36ff89c8214e,client_19581e27de,295097,low_ctr_visible_page,review_ctr,295097,7.3,page_1,0.05,104,keyword article,stable,0
8,9,content_8451fc6f034d,client_d029fa3a95,272144,low_ctr_visible_page,review_ctr,272144,2.3,top_3,0.03,20,keyword article,up,0
9,10,content_008fb02c46cb,client_349c41201b,236803,low_ctr_visible_page,review_ctr,236803,4.4,page_1,0.26,20,keyword article,down,1


## 3. Top-10 review

For each of the top 10 by score: the action, one line on why it's there, one line on what
would make it wrong. Built from the actual ranked rows below, so it stays honest about
whatever the rule really produces — not a hand-picked example.

In [ ]:
top10 = queue.head(10).copy()

def why_line(row):
    return (
        f"impressions_90d={row.impressions_90d:,.0f}, avg_position={row.avg_position:.1f} "
        f"({row.position_tier}), ctr={row.ctr:.2f}% — real search volume, a position where "
        "results get seen, but a click-through rate under the 0.5% cut."
    )

def wrong_if(row):
    notes = []
    if row.trend_direction != "down":
        notes.append(
            f"it's currently trending '{row.trend_direction}', not declining — this is an "
            "opportunity pick, not a decline rescue; wrong if the downstream use assumes "
            "every review_ctr row is already losing traffic."
        )
    if row.ctr < 0.10:
        notes.append(
            f"ctr is extremely low ({row.ctr:.2f}%) — worth a live SERP check first; a featured "
            "snippet, AI overview, or a broken title/meta tag above this result would explain "
            "the gap without any real problem in the article itself."
        )
    if not notes:
        notes.append(
            "if a SERP check shows a rich result already siphoning the click, the fix is the "
            "SERP feature, not this page."
        )
    return " ".join(notes)

for i, row in enumerate(top10.itertuples(index=False), 1):
    print(f"{i}. [{row.action} | {row.reason_code}] {row.content_id}")
    print(f"   why: {why_line(row)}")
    print(f"   wrong if: {wrong_if(row)}")
    print()

1. [review_ctr | low_ctr_visible_page] content_5fe46e04994d
   why: impressions_90d=517,715, avg_position=4.2 (page_1), ctr=0.14% — real search volume, a position where results get seen, but a click-through rate under the 0.5% cut.
   wrong if: if a SERP check shows a rich result already siphoning the click, the fix is the SERP feature, not this page.

2. [review_ctr | low_ctr_visible_page] content_aaef01a50def
   why: impressions_90d=517,109, avg_position=5.4 (page_1), ctr=0.25% — real search volume, a position where results get seen, but a click-through rate under the 0.5% cut.
   wrong if: it's currently trending 'stable', not declining — this is an opportunity pick, not a decline rescue; wrong if the downstream use assumes every review_ctr row is already losing traffic.

3. [review_ctr | low_ctr_visible_page] content_8c19996aa890
   why: impressions_90d=509,252, avg_position=2.5 (top_3), ctr=0.15% — real search volume, a position where results get seen, but a click-through rate und

## 4. Weak picks + leakage check

Which of the top 10 look weakest, and the check that no future-window, label-derived, or
product-flag column made it into the score.

In [ ]:
print("Weak picks in the top 10:")
print(
    "- Rows where trend_direction != 'down' (see the 'wrong if' notes above): the rule's job is "
    "'low CTR for a visible page,' not 'declining page' — real, but a different job than a "
    "decline-rescue queue, and easy to conflate downstream.\n"
    "- All 10 top rows share content_type == 'keyword article'. That could be a genuine pattern "
    "(keyword articles carry the most impressions_90d in this slice) or the rule silently "
    "favoring one content type over others — worth checking precision within each content_type "
    "before trusting the ranking equally across all of them."
)

# Leakage check: confirm the score only used pre-decision, trailing-window columns —
# never trend_direction/trend_pct (the label's own source), a *_last_30d/*_prev_30d
# comparison window, or a product decision flag.
score_inputs = {"impressions_90d", "avg_position", "ctr"}
label_source_cols = {"trend_direction", "trend_pct", "is_declining_label"}
future_window_cols = {c for c in df.columns if c.endswith("_last_30d") or c.endswith("_prev_30d")}
product_flag_cols = {"health_score", "quick_win_flag", "needs_attention_flag"}

print("\nScore inputs:", score_inputs)
print("Overlap with label-derived columns:", score_inputs & label_source_cols)
print("Overlap with future/comparison-window columns:", score_inputs & future_window_cols)
print("Product-flag columns present in this dataset at all:", product_flag_cols & set(df.columns))

assert not (score_inputs & label_source_cols)
assert not (score_inputs & future_window_cols)
assert not (product_flag_cols & set(df.columns))
print("\nClean: no label-derived, future-window, or product-flag column feeds the score.")

Weak picks in the top 10:
- Rows where trend_direction != 'down' (see the 'wrong if' notes above): the rule's job is 'low CTR for a visible page,' not 'declining page' — real, but a different job than a decline-rescue queue, and easy to conflate downstream.
- All 10 top rows share content_type == 'keyword article'. That could be a genuine pattern (keyword articles carry the most impressions_90d in this slice) or the rule silently favoring one content type over others — worth checking precision within each content_type before trusting the ranking equally across all of them.

Score inputs: {'impressions_90d', 'avg_position', 'ctr'}
Overlap with label-derived columns: set()
Overlap with future/comparison-window columns: set()
Product-flag columns present in this dataset at all: set()

Clean: no label-derived, future-window, or product-flag column feeds the score.


## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.